# FNSPID

## 0. Imports

In [1]:
import pathlib
import re
import time
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

try:
    import polars as pl
except Exception:
    pl = None

try:
    from langdetect import detect as langdetect_detect
except Exception:
    langdetect_detect = None


## 1. FNSPID Dataset Loading


In [2]:
# Cell: convert CSVs from the provided FNSPID path -> Parquet

# User-provided source path (update if needed)
SRC = pathlib.Path("/home/michaelschlee/ownCloud/GIT/LabelFusion/Dataset_Descriptives/data/FNSPID")
DST = pathlib.Path("data/Reuters commodities/monthly_series_parquet")

if not SRC.exists():
    print("Source path does not exist:", SRC)
else:
    csvs = sorted(SRC.glob("*.csv"))
    print(f"Found {len(csvs)} CSV files in {SRC}")

    for i, f in enumerate(csvs, 1):
        out = DST / f.with_suffix('.parquet').name
        try:
            if pl is None:
                raise ImportError("polars is not installed")
            pl.scan_csv(str(f)).with_columns(pl.col("value").cast(pl.Float64)).collect()
            print(f"[{i}/{len(csvs)}] would write {out.name} (parquet export skipped)")
        except Exception as e:
            try:
                it = pd.read_csv(f, parse_dates=["date"], usecols=["date","value"], chunksize=100000)
                parts = []
                for c in it:
                    c["value"] = pd.to_numeric(c["value"], errors="coerce")
                    parts.append(c)
                if parts:
                    df = pd.concat(parts)
                    print(f"[{i}/{len(csvs)}] would write {out.name} (parquet export skipped)")
                else:
                    print(f"[{i}/{len(csvs)}] no data in chunks: {f.name}")
            except Exception as e2:
                print(f"[{i}/{len(csvs)}] FAILED {f.name}: {e} / {e2}")
        time.sleep(0.05)

print('Done (cell is safe to run).')


Found 8 CSV files in /home/michaelschlee/ownCloud/GIT/LabelFusion/Dataset_Descriptives/data/FNSPID
[1/8] FAILED All_external.csv: unable to find column "value"; valid columns: ["Date", "Article_title", "Stock_symbol", "Url", "Publisher", "Author", "Article", "Lsa_summary", "Luhn_summary", "Textrank_summary", "Lexrank_summary"]

Resolved plan until failure:

	---> FAILED HERE RESOLVING 'with_columns' <---
Csv SCAN [/home/michaelschlee/ownCloud/GIT/LabelFusion/Dataset_Descriptives/data/FNSPID/All_external.csv]
PROJECT */11 COLUMNS
ESTIMATED ROWS: 32564756

This error occurred with the following context stack:
	[1] 'with_columns'
 / Usecols do not match columns, columns expected but not found: ['date', 'value']
[2/8] FAILED nasdaq_exteral_data.csv: unable to find column "value"; valid columns: ["Unnamed: 0", "Date", "Article_title", "Stock_symbol", "Url", "Publisher", "Author", "Article", "Lsa_summary", "Luhn_summary", "Textrank_summary", "Lexrank_summary"]

Resolved plan until failure:



In [3]:
# Cell: convert CSVs to Parquet (Polars preferred, Pandas fallback)

src = pathlib.Path("data/Reuters commodities/monthly_series")
dst = pathlib.Path("data/Reuters commodities/monthly_series_parquet")

csvs = list(src.glob("*.csv")) if src.exists() else []
print(f"Converting {len(csvs)} CSVs from {src} -> {dst}")

for i, f in enumerate(csvs, 1):
    out = dst / f.with_suffix(".parquet").name
    try:
        if pl is None:
            raise ImportError("polars is not installed")
        # use lazy scan to avoid loading everything at once
        pl.scan_csv(str(f)).with_columns(pl.col("value").cast(pl.Float64)).collect()
        print(f"[{i}/{len(csvs)}] would write {out.name} (parquet export skipped)")
    except Exception as e:
        # fallback to pandas in streaming/chunks if polars unavailable or fails
        try:
            it = pd.read_csv(f, parse_dates=["date"], usecols=["date","value"], chunksize=100000)
            parts = []
            for c in it:
                c["value"] = pd.to_numeric(c["value"], errors="coerce")
                parts.append(c)
            if parts:
                df = pd.concat(parts)
                print(f"[{i}/{len(csvs)}] would write {out.name} (parquet export skipped)")
            else:
                print(f"[{i}/{len(csvs)}] no data in chunks: {f.name}")
        except Exception as e2:
            print(f"[{i}/{len(csvs)}] FAILED {f.name}: {e} / {e2}")
    time.sleep(0.05)

print("Conversion done.")


Converting 76 CSVs from data/Reuters commodities/monthly_series -> data/Reuters commodities/monthly_series_parquet
[1/76] FAILED rand__EXSFUS.csv: unable to find column "value"; valid columns: ["date", "rand"]

Resolved plan until failure:

	---> FAILED HERE RESOLVING 'with_columns' <---
Csv SCAN [data/Reuters commodities/monthly_series/rand__EXSFUS.csv]
PROJECT */2 COLUMNS
ESTIMATED ROWS: 14

This error occurred with the following context stack:
	[1] 'with_columns'
 / Usecols do not match columns, columns expected but not found: ['value']
[2/76] FAILED cpi__CPIAUCSL.csv: unable to find column "value"; valid columns: ["date", "cpi"]

Resolved plan until failure:

	---> FAILED HERE RESOLVING 'with_columns' <---
Csv SCAN [data/Reuters commodities/monthly_series/cpi__CPIAUCSL.csv]
PROJECT */2 COLUMNS
ESTIMATED ROWS: 14

This error occurred with the following context stack:
	[1] 'with_columns'
 / Usecols do not match columns, columns expected but not found: ['value']
[3/76] FAILED rice__AP

## 2. Efficient Loading for Large CSV Files (20GB+)

For 20GB+ datasets, we need to use **lazy evaluation** and **streaming** approaches. This section demonstrates efficient loading strategies.


### 2.1 Data exploration

In [4]:
# FNSPID data source (absolute path)
FNSPID_PATH = pathlib.Path("/home/michaelschlee/ownCloud/GIT/LabelFusion/Dataset_Descriptives/data/FNSPID")

print("=== FNSPID DATA LOCATION ===")
print(f"Path: {FNSPID_PATH}")
print(f"Exists: {FNSPID_PATH.exists()}\n")

if FNSPID_PATH.exists():
    # List all CSV files
    csv_files = sorted(FNSPID_PATH.glob("*.csv"))
    print(f"CSV Files Found: {len(csv_files)}")
    for f in csv_files:
        size_gb = f.stat().st_size / (1024**3)
        print(f"  - {f.name:<30} {size_gb:>6.1f} GB")


=== FNSPID DATA LOCATION ===
Path: /home/michaelschlee/ownCloud/GIT/LabelFusion/Dataset_Descriptives/data/FNSPID
Exists: True

CSV Files Found: 8
  - All_external.csv                  5.3 GB
  - nasdaq_exteral_data.csv          21.6 GB
  - nasdaq_external_data_filtered_en.csv   10.9 GB
  - nasdaq_top10_2022_in_2023.csv     0.2 GB
  - test_df.csv                       0.1 GB
  - top10_2022.csv                    0.1 GB
  - top10_2023.csv                    0.2 GB
  - train_df.csv                      0.2 GB


In [5]:
# Method 1: LAZY LOADING from FNSPID CSVs (memory-efficient)
# Use this to query/filter data WITHOUT loading entire 20GB into RAM

FNSPID_PATH = pathlib.Path("/home/michaelschlee/ownCloud/GIT/LabelFusion/Dataset_Descriptives/data/FNSPID")
csv_files = sorted(FNSPID_PATH.glob("*.csv"))

if csv_files:
    print(f"Found {len(csv_files)} CSV file(s) in FNSPID\n")

    # Example: Load first CSV file lazily
    file_to_load = csv_files[0]
    print(f"Loading: {file_to_load.name} ({file_to_load.stat().st_size / (1024**3):.1f} GB)")
    print("Using lazy evaluation (NOT loading into memory yet)...\n")

    if pl is None:
        print("polars is not available in this environment.")
    else:
        lazy_df = pl.scan_csv(str(file_to_load))

        # Lazy queries - data is NOT loaded until .collect()
        result = (
            lazy_df
            .limit(1000)  # Take first 1000 rows for inspection
            .collect()  # NOW data is loaded into memory
        )

        print(f"Shape: {result.shape}")
        print(f"Columns: {result.columns}")
        print(f"\nFirst 5 rows:")
        print(result.head(5))
else:
    print("No CSV files found!")


Found 8 CSV file(s) in FNSPID

Loading: All_external.csv (5.3 GB)
Using lazy evaluation (NOT loading into memory yet)...

Shape: (1000, 11)
Columns: ['Date', 'Article_title', 'Stock_symbol', 'Url', 'Publisher', 'Author', 'Article', 'Lsa_summary', 'Luhn_summary', 'Textrank_summary', 'Lexrank_summary']

First 5 rows:
shape: (5, 11)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ Date      ┆ Article_t ┆ Stock_sym ┆ Url       ┆ … ┆ Lsa_summa ┆ Luhn_summ ┆ Textrank_ ┆ Lexrank_ │
│ ---       ┆ itle      ┆ bol       ┆ ---       ┆   ┆ ry        ┆ ary       ┆ summary   ┆ summary  │
│ str       ┆ ---       ┆ ---       ┆ str       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆ str       ┆ str       ┆           ┆   ┆ str       ┆ str       ┆ str       ┆ str      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 2020-06-0 ┆ Stocks    ┆ A         ┆ https://w ┆ … ┆ null    

In [6]:
# Method 2: INSPECT LARGE FILE STRUCTURE (Peek without loading)
# Get metadata about the CSV without loading entire dataset

FNSPID_PATH = pathlib.Path("/home/michaelschlee/ownCloud/GIT/LabelFusion/Dataset_Descriptives/data/FNSPID")
csv_files = sorted(FNSPID_PATH.glob("*.csv"))

if csv_files:
    file_to_inspect = csv_files[-1]  # Load the largest file
    print(f"Inspecting: {file_to_inspect.name} ({file_to_inspect.stat().st_size / (1024**3):.1f} GB)\n")

    if pl is None:
        print("polars is not available in this environment.")
    else:
        # Scan only first chunk to get column info
        sample = pl.scan_csv(str(file_to_inspect)).limit(10).collect()

        print("Column Names & Types:")
        for col_name, col_type in zip(sample.columns, sample.dtypes):
            print(f"  - {col_name:<20} {col_type}")

        print(f"\nTotal columns: {len(sample.columns)}")
        print(f"Sample data (first 10 rows):")
        print(sample)
else:
    print("No CSV files found!")


Inspecting: train_df.csv (0.2 GB)

Column Names & Types:
  - Date                 String
  - Article_title        String
  - label                String
  - Url                  String
  - Publisher            String
  - Author               String
  - Article              String
  - text                 String

Total columns: 8
Sample data (first 10 rows):
shape: (10, 8)
┌─────────────┬─────────────┬───────┬─────────────┬───────────┬────────┬─────────────┬─────────────┐
│ Date        ┆ Article_tit ┆ label ┆ Url         ┆ Publisher ┆ Author ┆ Article     ┆ text        │
│ ---         ┆ le          ┆ ---   ┆ ---         ┆ ---       ┆ ---    ┆ ---         ┆ ---         │
│ str         ┆ ---         ┆ str   ┆ str         ┆ str       ┆ str    ┆ str         ┆ str         │
│             ┆ str         ┆       ┆             ┆           ┆        ┆             ┆             │
╞═════════════╪═════════════╪═══════╪═════════════╪═══════════╪════════╪═════════════╪═════════════╡
│ 2022-12-31  ┆ Whe

In [7]:
# Method 3: PROCESS LARGE FNSPID DATA IN CHUNKS
# This keeps memory bounded by reading only one chunk at a time.

FNSPID_PATH = pathlib.Path("/home/michaelschlee/ownCloud/GIT/LabelFusion/Dataset_Descriptives/data/FNSPID")
csv_files = sorted(FNSPID_PATH.glob("*.csv"))

def analyze_file_in_chunks(file_path, chunk_size=50_000, max_chunks=5):
    """Stream through a large CSV and collect lightweight stats."""
    print(f"Analyzing {file_path.name} ({file_path.stat().st_size / (1024**3):.1f} GB)")
    print(f"Processing in {chunk_size:,}-row chunks...\n")

    total_rows = 0
    chunk_count = 0
    reader = pd.read_csv(file_path, chunksize=chunk_size)

    for chunk in reader:
        chunk_count += 1
        rows = len(chunk)
        total_rows += rows

        preview_cols = list(chunk.columns[:5])
        print(f"  Chunk {chunk_count}: {rows:>8,} rows | Total so far: {total_rows:>10,} | Columns: {preview_cols}")

        # Drop the chunk immediately so memory can be reclaimed.
        del chunk

        if chunk_count >= max_chunks:
            print("  ... stopping after a few chunks to avoid unnecessary memory use")
            break

    return total_rows, chunk_count

if csv_files:
    largest_file = max(csv_files, key=lambda f: f.stat().st_size)
    total, chunks_seen = analyze_file_in_chunks(largest_file, chunk_size=50_000, max_chunks=5)
    print(f"\n✓ Processed {chunks_seen} chunk(s); observed ~{total:,} rows so far")
else:
    print("No CSV files found!")


Analyzing nasdaq_exteral_data.csv (21.6 GB)
Processing in 50,000-row chunks...

  Chunk 1:   50,000 rows | Total so far:     50,000 | Columns: ['Unnamed: 0', 'Date', 'Article_title', 'Stock_symbol', 'Url']
  Chunk 2:   50,000 rows | Total so far:    100,000 | Columns: ['Unnamed: 0', 'Date', 'Article_title', 'Stock_symbol', 'Url']
  Chunk 3:   50,000 rows | Total so far:    150,000 | Columns: ['Unnamed: 0', 'Date', 'Article_title', 'Stock_symbol', 'Url']
  Chunk 4:   50,000 rows | Total so far:    200,000 | Columns: ['Unnamed: 0', 'Date', 'Article_title', 'Stock_symbol', 'Url']
  Chunk 5:   50,000 rows | Total so far:    250,000 | Columns: ['Unnamed: 0', 'Date', 'Article_title', 'Stock_symbol', 'Url']
  ... stopping after a few chunks to avoid unnecessary memory use

✓ Processed 5 chunk(s); observed ~250,000 rows so far


In [8]:
# Preview the first 5 rows from the largest FNSPID CSV without loading the full file

FNSPID_PATH = pathlib.Path("/home/michaelschlee/ownCloud/GIT/LabelFusion/Dataset_Descriptives/data/FNSPID")
csv_files = sorted(FNSPID_PATH.glob("*.csv"))

if csv_files:
    largest_file = max(csv_files, key=lambda f: f.stat().st_size)
    print(f"Previewing first 5 rows from: {largest_file.name}")
    preview = pd.read_csv(largest_file, nrows=5)
    print(preview)
else:
    print("No CSV files found in", FNSPID_PATH)


Previewing first 5 rows from: nasdaq_exteral_data.csv
   Unnamed: 0                     Date  \
0         0.0  2023-12-16 23:00:00 UTC   
1         1.0  2023-12-12 00:00:00 UTC   
2         2.0  2023-12-12 00:00:00 UTC   
3         3.0  2023-12-07 00:00:00 UTC   
4         4.0  2023-12-07 00:00:00 UTC   

                                       Article_title Stock_symbol  \
0  Interesting A Put And Call Options For August ...            A   
1  Wolfe Research Initiates Coverage of Agilent T...            A   
2  Agilent Technologies Reaches Analyst Target Price            A   
3  Agilent (A) Enhances BioTek Cytation C10 With ...            A   
4  Pre-Market Most Active for Dec 7, 2023 : SQQQ,...            A   

                                                 Url  Publisher  Author  \
0  https://www.nasdaq.com/articles/interesting-a-...        NaN     NaN   
1  https://www.nasdaq.com/articles/wolfe-research...        NaN     NaN   
2  https://www.nasdaq.com/articles/agilent-techno... 

In [9]:
# Show the columns from the largest FNSPID CSV without loading the full file

FNSPID_PATH = pathlib.Path("/home/michaelschlee/ownCloud/GIT/LabelFusion/Dataset_Descriptives/data/FNSPID")
csv_files = sorted(FNSPID_PATH.glob("*.csv"))

if csv_files:
    largest_file = max(csv_files, key=lambda f: f.stat().st_size)
    columns = pd.read_csv(largest_file, nrows=0).columns.tolist()
    print(f"Columns in {largest_file.name}:")
    for i, column in enumerate(columns, 1):
        print(f"{i}. {column}")
else:
    print("No CSV files found in", FNSPID_PATH)


Columns in nasdaq_exteral_data.csv:
1. Unnamed: 0
2. Date
3. Article_title
4. Stock_symbol
5. Url
6. Publisher
7. Author
8. Article
9. Lsa_summary
10. Luhn_summary
11. Textrank_summary
12. Lexrank_summary


In [10]:
# Show the columns from All_external.csv without loading the full file

FNSPID_PATH = pathlib.Path("/home/michaelschlee/ownCloud/GIT/LabelFusion/Dataset_Descriptives/data/FNSPID")
file_path = FNSPID_PATH / "All_external.csv"

if file_path.exists():
    columns = pd.read_csv(file_path, nrows=0).columns.tolist()
    print(f"Columns in {file_path.name}:")
    for i, column in enumerate(columns, 1):
        print(f"{i}. {column}")
else:
    print("File not found:", file_path)


Columns in All_external.csv:
1. Date
2. Article_title
3. Stock_symbol
4. Url
5. Publisher
6. Author
7. Article
8. Lsa_summary
9. Luhn_summary
10. Textrank_summary
11. Lexrank_summary


In [11]:
# Show the columns from nasdaq_exteral_data.csv without loading the full file

FNSPID_PATH = pathlib.Path("/home/michaelschlee/ownCloud/GIT/LabelFusion/Dataset_Descriptives/data/FNSPID")
file_path = FNSPID_PATH / "nasdaq_exteral_data.csv"

if file_path.exists():
    columns = pd.read_csv(file_path, nrows=0).columns.tolist()
    print(f"Columns in {file_path.name}:")
    for i, column in enumerate(columns, 1):
        print(f"{i}. {column}")
else:
    print("File not found:", file_path)


Columns in nasdaq_exteral_data.csv:
1. Unnamed: 0
2. Date
3. Article_title
4. Stock_symbol
5. Url
6. Publisher
7. Author
8. Article
9. Lsa_summary
10. Luhn_summary
11. Textrank_summary
12. Lexrank_summary


### 2.2 Statistics

In [12]:
# # Year-wise summary statistics for Stock_symbol (memory-safe chunked processing)

# FNSPID_PATH = pathlib.Path("/home/michaelschlee/ownCloud/GIT/LabelFusion/Dataset_Descriptives/data/FNSPID")
# file_path = FNSPID_PATH / "nasdaq_exteral_data.csv"

# if not file_path.exists():
#     print("File not found:", file_path)
# else:
#     chunk_size = 100_000
#     year_rows = Counter()
#     year_symbol_counts = defaultdict(Counter)

#     print(f"Analyzing {file_path.name} in {chunk_size:,}-row chunks...")

#     reader = pd.read_csv(
#         file_path,
#         usecols=["Date", "Stock_symbol"],
#         dtype={"Date": "string", "Stock_symbol": "string"},
#         chunksize=chunk_size,
#     )

#     for chunk in reader:
#         years = chunk["Date"].str.slice(0, 4)
#         symbols = chunk["Stock_symbol"].fillna("UNKNOWN")

#         for year, symbol in zip(years, symbols):
#             if pd.isna(year):
#                 continue
#             year = str(year)
#             year_rows[year] += 1
#             year_symbol_counts[year][str(symbol)] += 1

#         del chunk

#     summary_rows = []
#     for year in sorted(year_rows.keys()):
#         symbol_counter = year_symbol_counts[year]
#         top_symbol, top_symbol_count = symbol_counter.most_common(1)[0]
#         summary_rows.append({
#             "year": year,
#             "rows": year_rows[year],
#             "unique_stock_symbols": len(symbol_counter),
#             "most_common_symbol": top_symbol,
#             "most_common_symbol_count": top_symbol_count,
#             "top_5_symbols": symbol_counter.most_common(5),
#         })

#     summary = pd.DataFrame(summary_rows)
#     print("Year-wise Stock_symbol summary:\n")
#     print(summary[["year", "rows", "unique_stock_symbols", "most_common_symbol", "most_common_symbol_count"]].to_string(index=False))

#     print("\nTop 5 symbols per year:")
#     for _, row in summary.iterrows():
#         print(f"{row['year']}: {row['top_5_symbols']}")

### 2.3 Data Filtering

In [13]:
# Filter FNSPID rows: known stock symbol, English article text, and non-empty article

FNSPID_PATH = pathlib.Path("/home/michaelschlee/ownCloud/GIT/LabelFusion/Dataset_Descriptives/data/FNSPID")
input_file = FNSPID_PATH / "nasdaq_exteral_data.csv"
output_file = FNSPID_PATH / "nasdaq_external_data_filtered_en.csv"

# Check if output file already exists; if so, skip filtering
if output_file.exists():
    print(f"✓ Cached file found: {output_file.name}")
    print(f"  Loading from disk instead of recomputing...")
    df = pd.read_csv(output_file, dtype='string')
    print(f"  Loaded {len(df):,} rows")
elif not input_file.exists():
    print("File not found:", input_file)
else:
    chunk_size = 50_000
    total_rows = 0
    kept_rows = 0

    # Optional language detector; falls back to a lightweight heuristic.
    if langdetect_detect is not None:
        def is_english_text(text: str) -> bool:
            try:
                return langdetect_detect(text) == "en"
            except Exception:
                return False

        detector_name = "langdetect"
    else:
        common_en = {
            "the", "and", "of", "to", "in", "for", "on", "with", "is", "that",
            "as", "by", "from", "at", "an", "be", "this", "it", "are", "was",
        }
        token_re = re.compile(r"[A-Za-z]+")

        def is_english_text(text: str) -> bool:
            tokens = [t.lower() for t in token_re.findall(text)]
            if len(tokens) < 8:
                return False
            score = sum(t in common_en for t in tokens) / max(len(tokens), 1)
            return score >= 0.05

        detector_name = "fallback_heuristic"

    first_chunk = True

    # Read only necessary columns to keep memory bounded.
    reader = pd.read_csv(
        input_file,
        usecols=["Date", "Article_title", "Stock_symbol", "Article", "Url", "Publisher", "Author"],
        dtype="string",
        chunksize=chunk_size,
        low_memory=True,
    )

    for chunk_idx, chunk in enumerate(reader, start=1):
        total_rows += len(chunk)

        # 1) Known stock symbol (not null/blank/unknown-like markers)
        symbol = chunk["Stock_symbol"].fillna("").str.strip()
        has_known_symbol = (
            symbol.ne("")
            & ~symbol.str.lower().isin({"unknown", "unk", "none", "null", "nan", "na"})
        )

        # 2) Article present and not empty
        article = chunk["Article"].fillna("").str.strip()
        has_article = article.ne("")

        # 3) English article text
        enough_length = article.str.len().ge(30)
        candidates = article[has_article & enough_length]
        english_mask = pd.Series(False, index=chunk.index)
        if not candidates.empty:
            english_mask.loc[candidates.index] = candidates.apply(is_english_text)

        filtered = chunk[has_known_symbol & has_article & english_mask].copy()
        kept_rows += len(filtered)

        filtered.to_csv(output_file, mode="w" if first_chunk else "a", index=False, header=first_chunk)
        first_chunk = False

        print(
            f"Chunk {chunk_idx}: input={len(chunk):,} | kept={len(filtered):,} | "
            f"running_kept={kept_rows:,}"
        )

        del chunk
        del filtered

    print("\nFiltering complete")
    print(f"Detector: {detector_name}")
    print(f"Input rows: {total_rows:,}")
    print(f"Kept rows: {kept_rows:,}")
    print(f"Output file: {output_file}")


✓ Cached file found: nasdaq_external_data_filtered_en.csv
  Loading from disk instead of recomputing...
  Loaded 2,491,699 rows


In [14]:
preview = pd.read_csv(output_file, nrows=5)
print(preview.to_string(index=False))

                   Date                                                                                            Article_title Stock_symbol                                                                                                                         Url  Publisher  Author                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            

In [15]:
# Stock symbol distribution for 2022 and 2023 from the filtered FNSPID export

filtered_file = Path("/home/michaelschlee/ownCloud/GIT/LabelFusion/Dataset_Descriptives/data/FNSPID/nasdaq_external_data_filtered_en.csv")

if not filtered_file.exists():
    print("Filtered file not found:", filtered_file)
else:
    chunk_size = 50_000
    target_years = {"2022", "2023"}
    year_symbol_counts = {year: Counter() for year in target_years}
    year_row_counts = Counter()

    reader = pd.read_csv(
        filtered_file,
        usecols=["Date", "Stock_symbol"],
        dtype={"Date": "string", "Stock_symbol": "string"},
        chunksize=chunk_size,
        low_memory=True,
    )

    for chunk in reader:
        years = chunk["Date"].str.slice(0, 4)
        symbols = chunk["Stock_symbol"].fillna("").str.strip()

        mask = years.isin(target_years) & symbols.ne("")
        if mask.any():
            subset = chunk.loc[mask, ["Date", "Stock_symbol"]].copy()
            subset["year"] = subset["Date"].str.slice(0, 4)

            for year, symbol in zip(subset["year"], subset["Stock_symbol"]):
                year_row_counts[str(year)] += 1
                year_symbol_counts[str(year)][str(symbol)] += 1

        del chunk

    for year in sorted(target_years):
        print(f"\n=== {year} ===")
        print(f"Total rows: {year_row_counts[year]:,}")
        if year_symbol_counts[year]:
            distribution = pd.DataFrame(
                year_symbol_counts[year].most_common(),
                columns=["Stock_symbol", "count"],
            )
            display(distribution.head(20))
        else:
            print("No rows found for this year.")


=== 2022 ===
Total rows: 280,345


,Stock_symbol,count
0,TSLA,3169
1,AAPL,3094
2,MSFT,2679
3,NVDA,2368
4,BRK,2364
5,XOM,1959
6,DIS,1922
7,WMT,1883
8,GOOG,1878
9,AMD,1864



=== 2023 ===
Total rows: 954,098


,Stock_symbol,count
0,BROGW,10456
1,BPYPO,9979
2,BHFAL,9614
3,PMAY,9108
4,ACGLO,9014
5,DNOV,8990
6,GSEE,8966
7,FDEV,8891
8,OCFCP,8832
9,PFFL,8821


### 2.4 Check for duplicates

## Data Sampling Procedure

To construct a balanced and representative subset of the FNSPID dataset for financial 
sentiment analysis, we applied a multi-step sampling procedure restricted to the 
Magnificent Seven (MAG7) companies: Apple (AAPL), Microsoft (MSFT), Nvidia (NVDA), 
Alphabet (GOOGL), Amazon (AMZN), Meta (META), and Tesla (TSLA).

### Phase 1 — Duplicate Detection

We first performed a duplicate check across all MAG7 articles from 2010 to 2023, 
identifying article pairs sharing identical normalized titles and body text within 
the same year. This step provided transparency about the extent of label overlap 
and cross-ticker duplication in the original FNSPID data, which is consistent with 
known labeling quality issues documented in prior work (Nordansjö et al. 2025).

### Phase 2 — Data Cleaning

We restricted the data to the period 2012–2023, representing the first year in which 
all seven companies had publicly traded stock and therefore meaningful news coverage. 
Historical ticker symbols were consolidated to their current equivalents 
(GOOG→GOOGL, FB→META) to ensure consistent 7-class encoding. Within-year duplicates 
identified in Phase 1 were removed.

### Phase 3 — Stratified Sampling

To ensure equal representation across both companies and time, we applied a 
two-dimensional stratified sampling scheme. A target of 22,000 articles in total 
was set, corresponding to approximately 3,143 articles per ticker and 262 articles 
per ticker per year across the 12-year window. Where a ticker/year combination 
contained fewer articles than the target, all available articles were retained. 
This approach preserves the natural co-occurrence distribution of tickers — that is, 
articles mentioning multiple MAG7 companies simultaneously — rather than artificially 
forcing a fixed single-vs-multi ratio, which would distort the realistic structure 
of financial news.

In [36]:
# Duplicate article-title/article-body check within each year: 2010 through 2023
# This checks whether the same article title AND article body appear more than once inside each year separately
# and shows which stock symbols are associated with each duplicate pair, filtered to the Mag7 set.
# After the duplicate check, clean data is sampled evenly across years (2012-2023) and tickers,
# with historical ticker consolidation and one-hot encoding of MAG7 tickers.
# META/FB has limited coverage in FNSPID due to sparse NASDAQ news feed collection;
# all available META articles are retained while other tickers are sampled to ~3,143 each.

from collections import defaultdict
from sklearn.preprocessing import MultiLabelBinarizer


def normalize_text(value: object) -> str:
    if pd.isna(value):
        return ""
    return " ".join(str(value).split()).strip().lower()


# ── Ticker setup ──
mag7 = ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'GOOG', 'AMZN', 'META', 'FB', 'TSLA']
mag7_set = {symbol.upper() for symbol in mag7}

# consolidate historical tickers to current ones
ticker_consolidation = {
    'GOOG': 'GOOGL',
    'FB': 'META',
}
# canonical 7 tickers for one-hot encoding
mag7_canonical = ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'AMZN', 'META', 'TSLA']

source_candidates = [
    Path("/home/michaelschlee/ownCloud/GIT/LabelFusion/Dataset_Descriptives/data/FNSPID/nasdaq_exteral_data.csv"),
    Path("/home/michaelschlee/ownCloud/GIT/LabelFusion/Dataset_Descriptives/data/FNSPID/nasdaq_external_data_filtered_en.csv"),
]
source_file = next((path for path in source_candidates if path.exists()), None)
if source_file is None:
    raise FileNotFoundError("Could not find a FNSPID CSV to inspect for duplicates")

print(f"Using source file: {source_file.name}")
print(f"Filtering to Mag7 symbols: {mag7}")

# ── Phase 0: Diagnostic — check raw ticker and year distribution ──
print("\n=== Phase 0: Diagnostic ===")

diag_rows = []
reader = pd.read_csv(
    source_file,
    usecols=lambda column: column in {"Date", "Article_title", "Article", "Stock_symbol"},
    dtype="string",
    chunksize=50_000,
    low_memory=True,
)

for chunk in reader:
    work = chunk.copy()
    work["year"] = work["Date"].fillna("").astype("string").str.slice(0, 4)
    work["Stock_symbol"] = work["Stock_symbol"].fillna("").astype("string").str.strip().str.upper()
    work = work.loc[work["Stock_symbol"].isin(mag7_set)].copy()
    if work.empty:
        continue
    diag_rows.append(work[["Stock_symbol", "year"]])

df_diag = pd.concat(diag_rows, ignore_index=True)

print(f"\nRaw ticker counts (before consolidation):")
print(df_diag["Stock_symbol"].value_counts().sort_index())

print(f"\nRaw year distribution across all MAG7 tickers (before consolidation):")
print(df_diag["year"].value_counts().sort_index())

print(f"\nRaw ticker counts per year (before consolidation):")
display(
    df_diag.groupby(["Stock_symbol", "year"])
    .size()
    .unstack(fill_value=0)
)

print(f"\nFB/META specific check:")
fb_meta = df_diag[df_diag["Stock_symbol"].isin(["FB", "META"])]
if fb_meta.empty:
    print("WARNING: No FB or META articles found in source file at all!")
else:
    print(fb_meta.groupby(["Stock_symbol", "year"]).size().unstack(fill_value=0))

print(f"\nGOOG/GOOGL specific check:")
goog = df_diag[df_diag["Stock_symbol"].isin(["GOOG", "GOOGL"])]
if goog.empty:
    print("WARNING: No GOOG or GOOGL articles found in source file at all!")
else:
    print(goog.groupby(["Stock_symbol", "year"]).size().unstack(fill_value=0))


# ── Phase 1: Duplicate check (2010-2023) ──
start_year = 2010
target_years = [str(year) for year in range(start_year, 2024)]
year_pair_stats = {year: defaultdict(lambda: {"count": 0, "stock_symbols": set()}) for year in target_years}

required_columns = {"Date", "Article_title", "Article", "Stock_symbol"}
chunk_size = 50_000

reader = pd.read_csv(
    source_file,
    usecols=lambda column: column in required_columns,
    dtype="string",
    chunksize=chunk_size,
    low_memory=True,
)

rows_seen = 0
rows_used = 0

for chunk in reader:
    rows_seen += len(chunk)

    if not required_columns.issubset(chunk.columns):
        missing = sorted(required_columns - set(chunk.columns))
        raise KeyError(f"Missing required columns in {source_file.name}: {missing}")

    work = chunk[["Date", "Article_title", "Article", "Stock_symbol"]].copy()
    work["year"] = work["Date"].fillna("").astype("string").str.slice(0, 4)
    work = work.loc[work["year"].isin(target_years)].copy()
    if work.empty:
        continue

    work["norm_title"] = work["Article_title"].fillna("").map(normalize_text)
    work["norm_article"] = work["Article"].fillna("").map(normalize_text)
    work["Stock_symbol"] = work["Stock_symbol"].fillna("").astype("string").str.strip().str.upper()
    work = work.loc[
        work["norm_title"].ne("")
        & work["norm_article"].ne("")
        & work["Stock_symbol"].ne("")
        & work["Stock_symbol"].isin(mag7_set)
    ].copy()
    if work.empty:
        continue

    rows_used += len(work)

    chunk_summary = (
        work.groupby(["year", "norm_title", "norm_article"], as_index=False)
        .agg(
            count=("Stock_symbol", "size"),
            stock_symbols=("Stock_symbol", lambda s: sorted(set(s.astype(str)))),
        )
    )

    for row in chunk_summary.itertuples(index=False):
        entry = year_pair_stats[str(row.year)][(row.norm_title, row.norm_article)]
        entry["count"] += int(row.count)
        entry["stock_symbols"].update(row.stock_symbols)

for year in target_years:
    print(f"\n=== {year} ===")

    pair_stats = year_pair_stats[year]
    if not pair_stats:
        print("No Mag7 rows found for this year.")
        continue

    summary_rows = []
    for (norm_title, norm_article), stats in pair_stats.items():
        if stats["count"] > 1:
            summary_rows.append(
                {
                    "normalized_article_title": norm_title,
                    "normalized_article": norm_article,
                    "count": stats["count"],
                    "stock_symbols": ", ".join(sorted(stats["stock_symbols"])),
                }
            )

    total_rows = sum(stats["count"] for stats in pair_stats.values())
    unique_pairs = len(pair_stats)
    duplicate_pairs = len(summary_rows)
    duplicate_rows = sum(item["count"] for item in summary_rows)

    print(f"Total Mag7 rows with non-empty title/article: {total_rows:,}")
    print(f"Unique Mag7 title/article pairs: {unique_pairs:,}")
    print(f"Duplicate Mag7 title/article pairs: {duplicate_pairs:,}")
    print(f"Rows involved in duplicate pairs: {duplicate_rows:,}")

    if summary_rows:
        display(
            pd.DataFrame(summary_rows)
            .sort_values("count", ascending=False)
            .head(20)
        )
    else:
        print("No duplicate Mag7 title/article pairs found in this year.")

print(f"\nProcessed {rows_seen:,} raw rows and used {rows_used:,} Mag7 rows across years {start_year}-2023.")


# ── Phase 2: Clean data collection (2012-2023) ──
print("\n\n=== Phase 2: Collecting clean data for sampling (2012-2023) ===")

target_years_sample = [str(year) for year in range(2012, 2024)]
clean_rows = []

reader = pd.read_csv(
    source_file,
    usecols=lambda column: column in required_columns,
    dtype="string",
    chunksize=chunk_size,
    low_memory=True,
)

for chunk in reader:
    work = chunk[["Date", "Article_title", "Article", "Stock_symbol"]].copy()
    work["year"] = work["Date"].fillna("").astype("string").str.slice(0, 4)
    work = work.loc[work["year"].isin(target_years_sample)].copy()
    if work.empty:
        continue

    work["norm_title"] = work["Article_title"].fillna("").map(normalize_text)
    work["norm_article"] = work["Article"].fillna("").map(normalize_text)
    work["Stock_symbol"] = work["Stock_symbol"].fillna("").astype("string").str.strip().str.upper()
    work = work.loc[
        work["norm_title"].ne("")
        & work["norm_article"].ne("")
        & work["Stock_symbol"].ne("")
        & work["Stock_symbol"].isin(mag7_set)
    ].copy()
    if work.empty:
        continue

    # consolidate historical tickers
    work["Stock_symbol"] = work["Stock_symbol"].replace(ticker_consolidation)

    # drop duplicates (same title + body within same year)
    work = work.drop_duplicates(subset=["year", "norm_title", "norm_article"])

    clean_rows.append(work)

df_clean = pd.concat(clean_rows, ignore_index=True)
print(f"Clean MAG7 rows available for sampling: {len(df_clean):,}")

print(f"\nAvailable articles per ticker (after consolidation, before sampling):")
print(df_clean["Stock_symbol"].value_counts().sort_index())

print(f"\nAvailable articles per ticker per year (after consolidation, before sampling):")
display(
    df_clean.groupby(["Stock_symbol", "year"])
    .size()
    .unstack(fill_value=0)
)


# ── Phase 3: Sampling per ticker only ──
# META/FB has limited coverage in FNSPID due to sparse NASDAQ news feed collection.
# All available META articles are retained; other tickers are sampled to ~3,143 each.
print("\n=== Phase 3: Sampling ===")

target_total = 22_000
n_tickers = len(mag7_canonical)                 # 7
target_per_ticker = target_total // n_tickers   # ~3,143

print(f"Target total articles: {target_total:,}")
print(f"Target per ticker (well-covered tickers): {target_per_ticker:,}")
print(f"META: retaining all available articles")

def sample_ticker(group, target):
    if group.name == "META":
        return group  # retain all META articles
    return group.sample(n=min(len(group), target), random_state=42)

df_sampled = (
    df_clean
    .groupby("Stock_symbol", group_keys=False)
    .apply(lambda x: sample_ticker(x, target_per_ticker))
    .reset_index(drop=True)
)

print(f"\nTotal articles sampled: {len(df_sampled):,}")
print(f"\nActual articles per ticker:")
print(df_sampled["Stock_symbol"].value_counts().sort_index())
print(f"\nActual articles per ticker per year:")
display(
    df_sampled.groupby(["Stock_symbol", "year"])
    .size()
    .unstack(fill_value=0)
)


# ── Phase 4: Collect all MAG7 tickers per article and one-hot encode ──
print("\n=== Phase 4: One-hot encoding ===")

# for each unique article, collect all MAG7 tickers it appears under in the sample
article_tickers = (
    df_sampled
    .groupby("norm_title")["Stock_symbol"]
    .apply(lambda s: sorted(set(s)))
    .reset_index()
    .rename(columns={"Stock_symbol": "tickers"})
)

df_sampled = df_sampled.merge(article_tickers, on="norm_title", how="left")

# one-hot encode using canonical 7 tickers
mlb = MultiLabelBinarizer(classes=sorted(mag7_canonical))
ohe = mlb.fit_transform(df_sampled["tickers"])
df_ohe = pd.DataFrame(ohe, columns=mlb.classes_, index=df_sampled.index)
df_final = pd.concat([df_sampled, df_ohe], axis=1)

# report natural co-occurrence distribution
df_final["n_mag7_tickers"] = df_ohe.sum(axis=1)
print(f"\nTicker co-occurrence counts (total articles per ticker):")
print(df_ohe.sum().sort_values(ascending=False))
print(f"\nDistribution of articles by number of MAG7 tickers mentioned:")
print(df_final["n_mag7_tickers"].value_counts().sort_index())
print(f"\nArticles mentioning multiple MAG7 tickers: {df_final['n_mag7_tickers'].gt(1).sum():,}")


# ── Phase 5: Summary statistics text ──
print("\n=== Phase 5: Summary Statistics ===")

total_articles = len(df_final)
ticker_counts = df_final["Stock_symbol"].value_counts().sort_index()
year_counts = df_final["year"].value_counts().sort_index()
n_multi = df_final["n_mag7_tickers"].gt(1).sum()
n_single = df_final["n_mag7_tickers"].eq(1).sum()
pct_multi = n_multi / total_articles * 100
pct_single = n_single / total_articles * 100
ohe_sums = df_ohe.sum().sort_values(ascending=False)

summary = f"""
## Summary Statistics

### Dataset Overview
- **Total articles sampled:** {total_articles:,}
- **Time period:** {df_final["year"].min()} – {df_final["year"].max()}
- **Number of tickers:** {df_final["Stock_symbol"].nunique()}
- **Number of years covered:** {df_final["year"].nunique()}

### Articles per Ticker
- Note: META articles are underrepresented in FNSPID due to sparse NASDAQ news feed
  collection under the FB ticker; all available META articles are retained.
{chr(10).join(f"- **{ticker}:** {count:,}" for ticker, count in ticker_counts.items())}

### Articles per Year
{chr(10).join(f"- **{year}:** {count:,}" for year, count in year_counts.items())}

### Ticker Co-occurrence
- **Single-ticker articles:** {n_single:,} ({pct_single:.1f}%)
- **Multi-ticker articles:** {n_multi:,} ({pct_multi:.1f}%)

### One-Hot Encoding Distribution
{chr(10).join(f"- **{ticker}:** {int(count):,} articles" for ticker, count in ohe_sums.items())}
"""

print(summary)


# ── Phase 6: Save ──
output_path = source_file.parent / "fnspid_mag7_sampled.csv"
df_final.drop(columns=["norm_title", "norm_article"]).to_csv(output_path, index=False)
print(f"\nSaved dataset to {output_path}")

md_output_path = source_file.parent / "fnspid_mag7_summary_statistics.md"
with open(md_output_path, "w") as f:
    f.write(summary)
print(f"Saved summary statistics to {md_output_path}")

Using source file: nasdaq_exteral_data.csv
Filtering to Mag7 symbols: ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'GOOG', 'AMZN', 'META', 'FB', 'TSLA']

=== Phase 0: Diagnostic ===

Raw ticker counts (before consolidation):
Stock_symbol
AAPL      9338
AMZN      5060
FB         433
GOOG      9930
GOOGL     1754
MSFT      8737
NVDA     11862
TSLA     10587
Name: count, dtype: Int64

Raw year distribution across all MAG7 tickers (before consolidation):
year
2011      226
2012      183
2013      144
2014      118
2015      180
2016      337
2017      615
2018      871
2019     4149
2020     5405
2021     2183
2022    13188
2023    30102
Name: count, dtype: Int64

Raw ticker counts per year (before consolidation):


year,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
Stock_symbol,,,,,,,,,,,,,
AAPL,0,0,0,0,0,0,0,0,0,473,0,3094,5771
AMZN,0,0,0,0,0,0,0,0,0,278,0,0,4782
FB,0,0,0,0,0,0,0,0,0,433,0,0,0
GOOG,0,0,0,0,0,0,0,47,2141,2081,1472,1878,2311
GOOGL,0,0,0,0,0,0,0,298,767,689,0,0,0
MSFT,0,0,0,0,0,0,0,0,0,0,0,2679,6058
NVDA,226,183,144,118,180,337,615,526,508,309,711,2368,5637
TSLA,0,0,0,0,0,0,0,0,733,1142,0,3169,5543



FB/META specific check:
year          2020
Stock_symbol      
FB             433

GOOG/GOOGL specific check:
year          2018  2019  2020  2021  2022  2023
Stock_symbol                                    
GOOG            47  2141  2081  1472  1878  2311
GOOGL          298   767   689     0     0     0

=== 2010 ===
No Mag7 rows found for this year.

=== 2011 ===
No Mag7 rows found for this year.

=== 2012 ===
No Mag7 rows found for this year.

=== 2013 ===
No Mag7 rows found for this year.

=== 2014 ===
No Mag7 rows found for this year.

=== 2015 ===
No Mag7 rows found for this year.

=== 2016 ===
No Mag7 rows found for this year.

=== 2017 ===
No Mag7 rows found for this year.

=== 2018 ===
No Mag7 rows found for this year.

=== 2019 ===
Total Mag7 rows with non-empty title/article: 1,428
Unique Mag7 title/article pairs: 1,425
Duplicate Mag7 title/article pairs: 3
Rows involved in duplicate pairs: 6


,normalized_article_title,normalized_article,count,stock_symbols
0,a detailed look at trends in advertising reven...,"google, facebook, and amazon are some of the l...",2,GOOG
1,the next year won’t be as rocky for facebook s...,"from the earliest days of facebook (nasdaq:), ...",2,GOOG
2,"with right partners now in place, jd stock mig...",whenever jd.com (nasdaq:) is examined it’s com...,2,GOOG



=== 2020 ===
Total Mag7 rows with non-empty title/article: 1,637
Unique Mag7 title/article pairs: 1,629
Duplicate Mag7 title/article pairs: 8
Rows involved in duplicate pairs: 16


...

year,2019,2020,2021,2022,2023
Stock_symbol,,,,,
AAPL,0,0,0,3083,5624
AMZN,0,0,0,0,4628
GOOGL,1425,1629,1472,1877,2306
MSFT,0,0,0,2655,5905
NVDA,0,0,708,2360,5496
TSLA,0,0,0,3143,5394



=== Phase 3: Sampling ===
Target total articles: 22,000
Target per ticker (well-covered tickers): 3,142
META: retaining all available articles

Total articles sampled: 18,852

Actual articles per ticker:
Stock_symbol
AAPL     3142
AMZN     3142
GOOGL    3142
MSFT     3142
NVDA     3142
TSLA     3142
Name: count, dtype: Int64

Actual articles per ticker per year:


/tmp/ipykernel_121092/2762878474.py:270: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: sample_ticker(x, target_per_ticker))


year,2019,2020,2021,2022,2023
Stock_symbol,,,,,
AAPL,0,0,0,1106,2036
AMZN,0,0,0,0,3142
GOOGL,509,578,535,643,877
MSFT,0,0,0,962,2180
NVDA,0,0,251,855,2036
TSLA,0,0,0,1151,1991



=== Phase 4: One-hot encoding ===

Ticker co-occurrence counts (total articles per ticker):
MSFT     4492
AAPL     4428
AMZN     4405
NVDA     3979
GOOGL    3762
TSLA     3750
META        0
dtype: int64

Distribution of articles by number of MAG7 tickers mentioned:
n_mag7_tickers
1    14449
2     3104
3     1083
4      170
5       46
Name: count, dtype: int64

Articles mentioning multiple MAG7 tickers: 4,403

=== Phase 5: Summary Statistics ===

## Summary Statistics

### Dataset Overview
- **Total articles sampled:** 18,852
- **Time period:** 2019 – 2023
- **Number of tickers:** 6
- **Number of years covered:** 5

### Articles per Ticker
- Note: META articles are underrepresented in FNSPID due to sparse NASDAQ news feed
  collection under the FB ticker; all available META articles are retained.
- **AAPL:** 3,142
- **AMZN:** 3,142
- **GOOGL:** 3,142
- **MSFT:** 3,142
- **NVDA:** 3,142
- **TSLA:** 3,142

### Articles per Year
- **2019:** 509
- **2020:** 578
- **2021:** 786
- **2022:** 

In [35]:
all_ticker_counts = []
reader = pd.read_csv(
    source_file,
    usecols=["Stock_symbol"],
    dtype="string",
    chunksize=50_000,
    low_memory=True,
)

for chunk in reader:
    chunk["Stock_symbol"] = chunk["Stock_symbol"].fillna("").astype("string").str.strip().str.upper()
    all_ticker_counts.append(chunk["Stock_symbol"].value_counts())

ticker_summary = pd.concat(all_ticker_counts).groupby(level=0).sum().sort_values(ascending=False)

print(f"Total unique tickers in dataset: {len(ticker_summary):,}")
print(f"\nAll tickers by article count:")
pd.set_option("display.max_rows", None)
print(ticker_summary)
pd.reset_option("display.max_rows")

Total unique tickers in dataset: 8,553

All tickers by article count:
Stock_symbol
          9804627
GILD        12376
NVDA        11862
QQQ         11813
BABA        11625
WFC         11301
INTC        11157
MRK         10774
TSLA        10587
KO          10521
BROGW       10456
BPYPO        9979
GOOG         9930
DIS          9654
BHFAL        9614
MU           9605
T            9463
MS           9458
AAPL         9338
AMD          9209
PMAY         9108
CLSN         9106
ACGLO        9014
DNOV         8990
GSEE         8966
QCOM         8954
FDEV         8891
OCFCP        8832
PFFL         8822
CMCSA        8820
BRK          8797
FDX          8744
MSFT         8737
GS           8730
HCRB         8722
FFEB         8718
UEVM         8715
BSMR         8708
UCIB         8699
WLDR         8698
KJUL         8693
CVX          8688
WMT          8686
GE           8680
GDMA         8680
WTRE         8680
ARTLW        8677
FSMB         8662
CGRO         8640
HVT-A        8630
UFEB         8603

In [17]:
# Extract top-10 symbols from 2022 and filter 2023 rows to those symbols

filtered_file = Path("/home/michaelschlee/ownCloud/GIT/LabelFusion/Dataset_Descriptives/data/FNSPID/nasdaq_external_data_filtered_en.csv")
out_file = filtered_file.with_name("nasdaq_top10_2022_in_2023.csv")

# Check if output file already exists; if so, skip processing
if out_file.exists():
    print(f"✓ Cached file found: {out_file.name}")
    print(f"  Loading from disk instead of recomputing...")
    df = pd.read_csv(out_file, dtype='string')
    print(f"  Loaded {len(df):,} rows")
    years = df['Date'].str.slice(0, 4)
    counts23 = df[years == '2023']['Stock_symbol'].value_counts().to_dict()
    top10 = list(counts23.keys())[:10]
    print('Top-10 symbols in 2022:', top10)
    rows = [(s, counts23.get(s, 0)) for s in top10]
    display(pd.DataFrame(rows, columns=['Stock_symbol', 'count_in_2023']))
    print('Loaded filtered 2023 rows from:', out_file)
elif not filtered_file.exists():
    print("Filtered file not found:", filtered_file)
else:
    chunksize = 50_000

    # 1) compute top-10 for 2022
    counter22 = Counter()
    for chunk in pd.read_csv(filtered_file, usecols=['Date', 'Stock_symbol'], dtype='string', chunksize=chunksize, low_memory=True):
        years = chunk['Date'].str.slice(0,4)
        mask = years == '2022'
        if mask.any():
            syms = chunk.loc[mask, 'Stock_symbol'].fillna('').str.strip()
            for s in syms:
                if s:
                    counter22[s] += 1
    top10 = [s for s,_ in counter22.most_common(10)]
    print('Top-10 symbols in 2022:', top10)

    # 2) filter 2023 rows whose symbol is in top10 and write to CSV
    first = True
    counts23 = Counter()
    cols = None
    for chunk in pd.read_csv(filtered_file, dtype='string', chunksize=chunksize, low_memory=True):
        years = chunk['Date'].str.slice(0,4)
        mask_year = years == '2023'
        if not mask_year.any():
            continue
        sub = chunk.loc[mask_year].copy()
        sym = sub['Stock_symbol'].fillna('').str.strip()
        keep_mask = sym.isin(top10)
        if keep_mask.any():
            kept = sub.loc[keep_mask].copy()
            kept.to_csv(out_file, mode='w' if first else 'a', index=False, header=first)
            first = False
            for s in kept['Stock_symbol'].fillna('').str.strip():
                counts23[s] += 1
    # 3) show counts for the top10 in 2023 (preserve same order)
    rows = [(s, counts23[s]) for s in top10]
    df = pd.DataFrame(rows, columns=['Stock_symbol', 'count_in_2023'])
    display(df)
    print('Written filtered 2023 rows to:', out_file)


✓ Cached file found: nasdaq_top10_2022_in_2023.csv
  Loading from disk instead of recomputing...
  Loaded 35,845 rows
Top-10 symbols in 2022: ['MSFT', 'AAPL', 'NVDA', 'TSLA', 'BRK', 'GOOG', 'AMD', 'DIS', 'XOM', 'WMT']


,Stock_symbol,count_in_2023
0,MSFT,6058
1,AAPL,5771
2,NVDA,5637
3,TSLA,5543
4,BRK,2480
5,GOOG,2311
6,AMD,2160
7,DIS,2080
8,XOM,1978
9,WMT,1827


Loaded filtered 2023 rows from: /home/michaelschlee/ownCloud/GIT/LabelFusion/Dataset_Descriptives/data/FNSPID/nasdaq_top10_2022_in_2023.csv


In [18]:
filtered_2023_path = "/home/michaelschlee/ownCloud/GIT/LabelFusion/Dataset_Descriptives/data/FNSPID/nasdaq_top10_2022_in_2023.csv"
filtered_2023 = pd.read_csv(filtered_2023_path)

symbol_distribution = (
    filtered_2023["Stock_symbol"]
    .value_counts()
    .rename_axis("Stock_symbol")
    .reset_index(name="count")
)

print(f"Loaded {len(filtered_2023):,} filtered 2023 rows from {filtered_2023_path}")
display(symbol_distribution)


Loaded 35,845 filtered 2023 rows from /home/michaelschlee/ownCloud/GIT/LabelFusion/Dataset_Descriptives/data/FNSPID/nasdaq_top10_2022_in_2023.csv


,Stock_symbol,count
0,MSFT,6058
1,AAPL,5771
2,NVDA,5637
3,TSLA,5543
4,BRK,2480
5,GOOG,2311
6,AMD,2160
7,DIS,2080
8,XOM,1978
9,WMT,1827


In [19]:
# Save 2022 and 2023 datasets as top10_2022.csv and top10_2023.csv

fnspid_path = Path("/home/michaelschlee/ownCloud/GIT/LabelFusion/Dataset_Descriptives/data/FNSPID")
filtered_file = fnspid_path / "nasdaq_external_data_filtered_en.csv"
out_2022 = fnspid_path / "top10_2022.csv"
out_2023 = fnspid_path / "top10_2023.csv"

# Check if both output files already exist; if so, skip processing
if out_2022.exists() and out_2023.exists():
    print(f"✓ Cached files found: {out_2022.name} and {out_2023.name}")
    print(f"  Loading from disk instead of recomputing...")
    df_2022 = pd.read_csv(out_2022, dtype='string')
    df_2023 = pd.read_csv(out_2023, dtype='string')
    print(f"  Loaded 2022: {len(df_2022):,} rows | 2023: {len(df_2023):,} rows")
    top10 = df_2022['Stock_symbol'].unique().tolist()
    print(f'Top-10 symbols from 2022: {top10}')
    print(f'\nCached 2022 data from: {out_2022}')
    display(df_2022.groupby('Stock_symbol').size().reset_index(name='count_2022'))
    print(f'\nCached 2023 data from: {out_2023}')
    display(df_2023.groupby('Stock_symbol').size().reset_index(name='count_2023'))
elif not filtered_file.exists():
    print("Filtered file not found:", filtered_file)
else:
    chunksize = 50_000

    # 1) Identify top-10 symbols from 2022
    counter22 = Counter()
    for chunk in pd.read_csv(filtered_file, usecols=['Date', 'Stock_symbol'], dtype='string', chunksize=chunksize, low_memory=True):
        years = chunk['Date'].str.slice(0, 4)
        mask = years == '2022'
        if mask.any():
            syms = chunk.loc[mask, 'Stock_symbol'].fillna('').str.strip()
            for s in syms:
                if s:
                    counter22[s] += 1
    top10 = [s for s, _ in counter22.most_common(10)]
    print('Top-10 symbols from 2022:', top10)

    # 2) Extract and save 2022 rows for top-10 symbols
    first_2022 = True
    counts22 = Counter()
    for chunk in pd.read_csv(filtered_file, dtype='string', chunksize=chunksize, low_memory=True):
        years = chunk['Date'].str.slice(0, 4)
        mask_year = years == '2022'
        if not mask_year.any():
            continue
        sub = chunk.loc[mask_year].copy()
        sym = sub['Stock_symbol'].fillna('').str.strip()
        keep_mask = sym.isin(top10)
        if keep_mask.any():
            kept = sub.loc[keep_mask].copy()
            kept.to_csv(out_2022, mode='w' if first_2022 else 'a', index=False, header=first_2022)
            first_2022 = False
            for s in kept['Stock_symbol'].fillna('').str.strip():
                counts22[s] += 1

    print(f'\nSaved 2022 data to: {out_2022}')
    df_2022 = pd.DataFrame([(s, counts22[s]) for s in top10], columns=['Stock_symbol', 'count_2022'])
    display(df_2022)

    # 3) Extract and save 2023 rows for top-10 symbols
    first_2023 = True
    counts23 = Counter()
    for chunk in pd.read_csv(filtered_file, dtype='string', chunksize=chunksize, low_memory=True):
        years = chunk['Date'].str.slice(0, 4)
        mask_year = years == '2023'
        if not mask_year.any():
            continue
        sub = chunk.loc[mask_year].copy()
        sym = sub['Stock_symbol'].fillna('').str.strip()
        keep_mask = sym.isin(top10)
        if keep_mask.any():
            kept = sub.loc[keep_mask].copy()
            kept.to_csv(out_2023, mode='w' if first_2023 else 'a', index=False, header=first_2023)
            first_2023 = False
            for s in kept['Stock_symbol'].fillna('').str.strip():
                counts23[s] += 1

    print(f'\nSaved 2023 data to: {out_2023}')
    df_2023 = pd.DataFrame([(s, counts23[s]) for s in top10], columns=['Stock_symbol', 'count_2023'])
    display(df_2023)


✓ Cached files found: top10_2022.csv and top10_2023.csv
  Loading from disk instead of recomputing...
  Loaded 2022: 23,180 rows | 2023: 35,845 rows
Top-10 symbols from 2022: ['AAPL', 'AMD', 'BRK', 'DIS', 'GOOG', 'MSFT', 'NVDA', 'TSLA', 'WMT', 'XOM']

Cached 2022 data from: /home/michaelschlee/ownCloud/GIT/LabelFusion/Dataset_Descriptives/data/FNSPID/top10_2022.csv


,Stock_symbol,count_2022
0,AAPL,3094
1,AMD,1864
2,BRK,2364
3,DIS,1922
4,GOOG,1878
5,MSFT,2679
6,NVDA,2368
7,TSLA,3169
8,WMT,1883
9,XOM,1959



Cached 2023 data from: /home/michaelschlee/ownCloud/GIT/LabelFusion/Dataset_Descriptives/data/FNSPID/top10_2023.csv


,Stock_symbol,count_2023
0,AAPL,5771
1,AMD,2160
2,BRK,2480
3,DIS,2080
4,GOOG,2311
5,MSFT,6058
6,NVDA,5637
7,TSLA,5543
8,WMT,1827
9,XOM,1978


### 2.4 Data Splitting

In [20]:
fnspid_path = Path("/home/michaelschlee/ownCloud/GIT/LabelFusion/Dataset_Descriptives/data/FNSPID")
file_2022 = fnspid_path / "top10_2022.csv"
file_2023 = fnspid_path / "top10_2023.csv"

if not file_2022.exists():
    print("File not found:", file_2022)
elif not file_2023.exists():
    print("File not found:", file_2023)
else:
    top10_2022 = pd.read_csv(file_2022, dtype="string")
    top10_2023 = pd.read_csv(file_2023, dtype="string")

    print(f"Loaded {len(top10_2022):,} rows from {file_2022.name}")
    print(f"Loaded {len(top10_2023):,} rows from {file_2023.name}")

    display(top10_2022.head(5))
    display(top10_2023.head(5))


Loaded 23,180 rows from top10_2022.csv
Loaded 35,845 rows from top10_2023.csv


,Date,Article_title,Stock_symbol,Url,Publisher,Author,Article
0,2022-12-31 00:00:00 UTC,Where Will Apple Stock Be in 1 Year?,AAPL,https://www.nasdaq.com/articles/where-will-app...,<NA>,<NA>,Apple's (NASDAQ: AAPL) stock declined nearly 3...
1,2022-12-31 00:00:00 UTC,Where Will Unity Software Stock Be in 3 Years?,AAPL,https://www.nasdaq.com/articles/where-will-uni...,<NA>,<NA>,Unity Software (NYSE: U) attracted a stampede ...
2,2022-12-31 00:00:00 UTC,4 Red-Hot Growth Stocks to Buy in 2023 and Beyond,AAPL,https://www.nasdaq.com/articles/4-red-hot-grow...,<NA>,<NA>,"Inflation, rising interest rates, and other ma..."
3,2022-12-31 00:00:00 UTC,"Even in an Advertising Slowdown, These 3 Stock...",AAPL,https://www.nasdaq.com/articles/even-in-an-adv...,<NA>,<NA>,"Marketers are trimming their ad budgets, and t..."
4,2022-12-31 00:00:00 UTC,3 Unstoppable Growth Stocks to Buy After a Sto...,AAPL,https://www.nasdaq.com/articles/3-unstoppable-...,<NA>,<NA>,The stock market's 2022 sell-off has highlight...


,Date,Article_title,Stock_symbol,Url,Publisher,Author,Article
0,2023-12-16 22:00:00 UTC,My 6 Largest Portfolio Holdings Heading Into 2...,AAPL,https://www.nasdaq.com/articles/my-6-largest-p...,<NA>,<NA>,"After an absolute disaster of a year in 2022, ..."
1,2023-12-16 22:00:00 UTC,Brokers Suggest Investing in Apple (AAPL): Rea...,AAPL,https://www.nasdaq.com/articles/brokers-sugges...,<NA>,<NA>,"When deciding whether to buy, sell, or hold a ..."
2,2023-12-16 21:00:00 UTC,"Company News for Dec 19, 2023",AAPL,https://www.nasdaq.com/articles/company-news-f...,<NA>,<NA>,Shares of Apple Inc. AAPL lost 0.9% on China’s...
3,2023-12-16 21:00:00 UTC,NVIDIA (NVDA) Up 243% YTD: Will It Carry Momen...,AAPL,https://www.nasdaq.com/articles/nvidia-nvda-up...,<NA>,<NA>,NVIDIA Corporation NVDA has witnessed a remark...
4,2023-12-16 21:00:00 UTC,"Pre-Market Most Active for Dec 19, 2023 : BMY,...",AAPL,https://www.nasdaq.com/articles/pre-market-mos...,<NA>,<NA>,The NASDAQ 100 Pre-Market Indicator is up 10.1...


In [21]:
train_df = top10_2022.copy()
train_df

,Date,Article_title,Stock_symbol,Url,Publisher,Author,Article
0,2022-12-31 00:00:00 UTC,Where Will Apple Stock Be in 1 Year?,AAPL,https://www.nasdaq.com/articles/where-will-app...,<NA>,<NA>,Apple's (NASDAQ: AAPL) stock declined nearly 3...
1,2022-12-31 00:00:00 UTC,Where Will Unity Software Stock Be in 3 Years?,AAPL,https://www.nasdaq.com/articles/where-will-uni...,<NA>,<NA>,Unity Software (NYSE: U) attracted a stampede ...
2,2022-12-31 00:00:00 UTC,4 Red-Hot Growth Stocks to Buy in 2023 and Beyond,AAPL,https://www.nasdaq.com/articles/4-red-hot-grow...,<NA>,<NA>,"Inflation, rising interest rates, and other ma..."
3,2022-12-31 00:00:00 UTC,"Even in an Advertising Slowdown, These 3 Stock...",AAPL,https://www.nasdaq.com/articles/even-in-an-adv...,<NA>,<NA>,"Marketers are trimming their ad budgets, and t..."
4,2022-12-31 00:00:00 UTC,3 Unstoppable Growth Stocks to Buy After a Sto...,AAPL,https://www.nasdaq.com/articles/3-unstoppable-...,<NA>,<NA>,The stock market's 2022 sell-off has highlight...
...,...,...,...,...,...,...,...
23175,2022-01-03 00:00:00 UTC,Exxon Q4 earnings poised to exceed pre-pandemi...,XOM,https://www.nasdaq.com/articles/exxon-q4-earni...,<NA>,<NA>,"By Sabrina Valle\nHOUSTON, Jan 3 (Reuters) - E..."
23176,2022-01-03 00:00:00 UTC,Macroeconomic Headwinds To Weigh On Williams C...,XOM,https://www.nasdaq.com/articles/macroeconomic-...,<NA>,<NA>,The shares of Williams Companies (NYSE: WMB) s...
23177,2022-01-03 00:00:00 UTC,Wall Street climbs in upbeat start to 2022,XOM,https://www.nasdaq.com/articles/wall-street-cl...,<NA>,<NA>,By Bansari Mayur Kamdar and Shashank Nayar\nJa...
23178,2022-01-03 00:00:00 UTC,US STOCKS-S&P futures near record high in brig...,XOM,https://www.nasdaq.com/articles/us-stocks-sp-f...,<NA>,<NA>,By Bansari Mayur Kamdar\nJan 3 (Reuters) - S&P...


In [22]:
test_df, _ = train_test_split(
    top10_2023,
    train_size=5800,
    stratify=top10_2023['Stock_symbol'],
    random_state=42
)

In [23]:
test_df

,Date,Article_title,Stock_symbol,Url,Publisher,Author,Article
29620,2023-06-13 00:00:00 UTC,3 Surprising EV Stocks Surging on Tesla’s Supe...,TSLA,https://www.nasdaq.com/articles/3-surprising-e...,<NA>,<NA>,"InvestorPlace - Stock Market News, Stock Advic..."
1323,2023-10-01 00:00:00 UTC,Nasdaq Sell-Off: My Top 3 Beaten-Down Growth S...,AAPL,https://www.nasdaq.com/articles/nasdaq-sell-of...,<NA>,<NA>,The Nasdaq Composite has come roaring back in ...
13301,2023-08-21 00:00:00 UTC,3 Stock-Split Stocks Billionaires Are Selling ...,GOOG,https://www.nasdaq.com/articles/3-stock-split-...,<NA>,<NA>,There are a lot of ways to make money on Wall ...
26496,2023-01-01 00:00:00 UTC,Can Nvidia Help Walmart and Target Solve Their...,NVDA,https://www.nasdaq.com/articles/can-nvidia-hel...,<NA>,<NA>,Walmart (NYSE: WMT) and Target (NYSE: TGT) hav...
10248,2023-02-06 00:00:00 UTC,Warren Buffett More Than Doubles His Money on ...,BRK,https://www.nasdaq.com/articles/warren-buffett...,<NA>,<NA>,When Berkshire Hathaway (NYSE: BRK.A)(NYSE: BR...
...,...,...,...,...,...,...,...
20978,2023-12-10 00:00:00 UTC,This Will Be a Trillion-Dollar Industry by 203...,NVDA,https://www.nasdaq.com/articles/this-will-be-a...,<NA>,<NA>,Artificial intelligence (AI) has powered Nvidi...
28683,2023-07-31 00:00:00 UTC,"Notable Monday Option Activity: FSLR, DIS, TSLA",TSLA,https://www.nasdaq.com/articles/notable-monday...,<NA>,<NA>,Looking at options trading activity among comp...
12702,2023-11-20 00:00:00 UTC,Sam Altman's Firing Opens Huge Opportunity for...,GOOG,https://www.nasdaq.com/articles/sam-altmans-fi...,<NA>,<NA>,Sam Altman was fired from OpenAI on Friday nig...
14200,2023-04-05 00:00:00 UTC,A Stock That Could Go Up 10x in a Decade,GOOG,https://www.nasdaq.com/articles/a-stock-that-c...,<NA>,<NA>,Not every company has the potential to rise in...


In [24]:
test_symbol_distribution = (
    test_df["Stock_symbol"]
    .fillna("UNKNOWN")
    .astype("string")
    .str.strip()
    .replace("", "UNKNOWN")
    .value_counts()
    .rename_axis("Stock_symbol")
    .reset_index(name="count")
)

display(test_symbol_distribution)
print(f"Total test_df rows: {len(test_df):,}")
print(f"Unique Stock_symbol values: {test_symbol_distribution['Stock_symbol'].nunique():,}")


,Stock_symbol,count
0,MSFT,980
1,AAPL,934
2,NVDA,912
3,TSLA,897
4,BRK,401
5,GOOG,374
6,AMD,349
7,DIS,337
8,XOM,320
9,WMT,296


Total test_df rows: 5,800
Unique Stock_symbol values: 10


In [25]:
label_column = "Stock_symbol"

train_labels = set(train_df[label_column].dropna().unique())
test_labels = set(test_df[label_column].dropna().unique())

print(f"Same labels: {train_labels == test_labels}")
print(f"Train labels ({len(train_labels)}): {sorted(train_labels)}")
print(f"Test labels ({len(test_labels)}): {sorted(test_labels)}")
print(f"Only in train: {sorted(train_labels - test_labels)}")
print(f"Only in test: {sorted(test_labels - train_labels)}")

Same labels: True
Train labels (10): ['AAPL', 'AMD', 'BRK', 'DIS', 'GOOG', 'MSFT', 'NVDA', 'TSLA', 'WMT', 'XOM']
Test labels (10): ['AAPL', 'AMD', 'BRK', 'DIS', 'GOOG', 'MSFT', 'NVDA', 'TSLA', 'WMT', 'XOM']
Only in train: []
Only in test: []


In [26]:
from pathlib import Path
import pandas as pd

# Create a unified `text` column by concatenating `Article_title` and `Article`.
fnspid_path = Path('/home/michaelschlee/ownCloud/GIT/LabelFusion/Dataset_Descriptives/data/FNSPID')
train_path = fnspid_path / 'train_df.csv'
test_path = fnspid_path / 'test_df.csv'

def _fuse_text_col(df: pd.DataFrame) -> pd.DataFrame:
    # Prefer Article_title then Article; handle missing columns gracefully
    title_col = None
    article_col = None
    for c in ['Article_title', 'Article title', 'Title', 'article_title']:
        if c in df.columns:
            title_col = c
            break
    for c in ['Article', 'article', 'Body', 'body']:
        if c in df.columns:
            article_col = c
            break

    title_series = df[title_col].fillna('') if title_col is not None else pd.Series([''] * len(df))
    article_series = df[article_col].fillna('') if article_col is not None else pd.Series([''] * len(df))

    df['text'] = (title_series.astype(str).str.strip() + ' ' + article_series.astype(str).str.strip()).str.strip()
    return df

# Update in-memory DataFrames if available
updated = False
if 'train_df' in globals():
    train_df = _fuse_text_col(train_df)
    print(f"Added 'text' column to in-memory train_df ({len(train_df)} rows)")
    updated = True
elif train_path.exists():
    df_t = pd.read_csv(train_path, dtype='string')
    df_t = _fuse_text_col(df_t)
    df_t.to_csv(train_path, index=False)
    print(f"Added 'text' column and updated file: {train_path}")
    updated = True

if 'test_df' in globals():
    test_df = _fuse_text_col(test_df)
    print(f"Added 'text' column to in-memory test_df ({len(test_df)} rows)")
    updated = True
elif test_path.exists():
    df_ts = pd.read_csv(test_path, dtype='string')
    df_ts = _fuse_text_col(df_ts)
    df_ts.to_csv(test_path, index=False)
    print(f"Added 'text' column and updated file: {test_path}")
    updated = True

if not updated:
    print('No train/test DataFrame or saved CSV files found to update.')

Added 'text' column to in-memory train_df (23180 rows)
Added 'text' column to in-memory test_df (5800 rows)


In [27]:
from pathlib import Path

save_dir = Path('/home/michaelschlee/ownCloud/GIT/LabelFusion/Dataset_Descriptives/data/FNSPID')
save_dir.mkdir(parents=True, exist_ok=True)

train_df.to_csv(save_dir / 'train_df.csv', index=False)
test_df.to_csv(save_dir / 'test_df.csv', index=False)

print(f"Saved train_df ({len(train_df)} rows) to {save_dir / 'train_df.csv'}")
print(f"Saved test_df ({len(test_df)} rows) to {save_dir / 'test_df.csv'}")

Saved train_df (23180 rows) to /home/michaelschlee/ownCloud/GIT/LabelFusion/Dataset_Descriptives/data/FNSPID/train_df.csv
Saved test_df (5800 rows) to /home/michaelschlee/ownCloud/GIT/LabelFusion/Dataset_Descriptives/data/FNSPID/test_df.csv


## 3. Download Timeserieses

In [28]:
# Download daily timeseries for all labels in saved train/test splits
from pathlib import Path
import pandas as pd

try:
    import yfinance as yf
except Exception:
    yf = None

data_dir = Path('/home/michaelschlee/ownCloud/GIT/LabelFusion/Dataset_Descriptives/data/FNSPID')
train_path = data_dir / 'train_df.csv'
test_path = data_dir / 'test_df.csv'
out_dir = data_dir / 'timeseries_by_label'
out_dir.mkdir(parents=True, exist_ok=True)

# Common Yahoo ticker aliases for dataset symbols
symbol_aliases = {
    'BRK': ['BRK-B', 'BRK.A', 'BRK-B'],
    'BRK.B': ['BRK-B'],
    'BRK/A': ['BRK-A'],
}

def ticker_candidates(symbol: str) -> list[str]:
    symbol = str(symbol).strip()
    cands = [symbol]
    cands.extend(symbol_aliases.get(symbol, []))
    cands.append(symbol.replace('.', '-'))
    cands.append(symbol.replace('/', '-'))
    # Keep order but remove duplicates/empties
    seen = set()
    uniq = []
    for c in cands:
        c = c.strip()
        if c and c not in seen:
            uniq.append(c)
            seen.add(c)
    return uniq

if not train_path.exists() or not test_path.exists():
    print('Missing required files:')
    print(f'  train: {train_path} (exists={train_path.exists()})')
    print(f'  test : {test_path} (exists={test_path.exists()})')
elif yf is None:
    print("yfinance is not installed. Install it with: pip install yfinance")
else:
    train_df = pd.read_csv(train_path, dtype='string')
    test_df = pd.read_csv(test_path, dtype='string')

    # Use timeframe covered by the saved splits and extend by +/- 6 months
    all_df = pd.concat([train_df[['Date', 'Stock_symbol']], test_df[['Date', 'Stock_symbol']]], ignore_index=True)
    all_df['Date'] = pd.to_datetime(all_df['Date'], errors='coerce')
    all_df['Stock_symbol'] = all_df['Stock_symbol'].fillna('').str.strip()

    all_df = all_df[all_df['Date'].notna() & all_df['Stock_symbol'].ne('')].copy()

    if all_df.empty:
        print('No valid Date/Stock_symbol rows found in train/test files.')
    else:
        raw_start = all_df['Date'].min()
        raw_end = all_df['Date'].max()
        start_date = (raw_start - pd.DateOffset(months=6)).date().isoformat()
        end_date = (raw_end + pd.DateOffset(months=6)).date().isoformat()
        symbols = sorted(all_df['Stock_symbol'].unique().tolist())

        print(f'Symbols to download: {len(symbols)}')
        print(f'Base timeframe: {raw_start.date().isoformat()} -> {raw_end.date().isoformat()}')
        print(f'Extended timeframe (+/- 6 months): {start_date} -> {end_date}')
        print(f'Output folder: {out_dir}')

        failed = []
        saved = 0

        # yfinance end date is exclusive; add one day to include end_date.
        yf_end = (pd.to_datetime(end_date) + pd.Timedelta(days=1)).date().isoformat()

        for i, symbol in enumerate(symbols, 1):
            ts = None
            used_ticker = None
            last_err = None

            for ticker in ticker_candidates(symbol):
                try:
                    candidate = yf.download(
                        ticker,
                        start=start_date,
                        end=yf_end,
                        auto_adjust=False,
                        progress=False,
                        threads=False,
                    )
                    if candidate is not None and not candidate.empty:
                        ts = candidate
                        used_ticker = ticker
                        break
                except Exception as e:
                    last_err = e

            if ts is None or ts.empty:
                reason = str(last_err) if last_err else 'empty download'
                failed.append((symbol, reason))
                print(f'[{i}/{len(symbols)}] {symbol}: no data')
                continue

            ts = ts.reset_index()
            # Flatten potential MultiIndex columns
            ts.columns = [c[0] if isinstance(c, tuple) else c for c in ts.columns]
            ts['Stock_symbol'] = symbol
            ts['Downloaded_ticker'] = used_ticker

            out_file = out_dir / f'{symbol}.csv'
            ts.to_csv(out_file, index=False)
            saved += 1
            if used_ticker != symbol:
                print(f'[{i}/{len(symbols)}] {symbol}: saved {len(ts)} rows via {used_ticker} -> {out_file.name}')
            else:
                print(f'[{i}/{len(symbols)}] {symbol}: saved {len(ts)} rows -> {out_file.name}')

        print('\nDownload complete')
        print(f'Successfully saved: {saved}/{len(symbols)}')
        if failed:
            fail_df = pd.DataFrame(failed, columns=['Stock_symbol', 'reason'])
            fail_path = out_dir / 'failed_symbols.csv'
            fail_df.to_csv(fail_path, index=False)
            print(f'Failed symbols: {len(failed)} (saved to {fail_path.name})')

Symbols to download: 10
Base timeframe: 2022-01-01 -> 2023-12-30
Extended timeframe (+/- 6 months): 2021-07-01 -> 2024-06-30
Output folder: /home/michaelschlee/ownCloud/GIT/LabelFusion/Dataset_Descriptives/data/FNSPID/timeseries_by_label
[1/10] AAPL: saved 753 rows -> AAPL.csv
[2/10] AMD: saved 753 rows -> AMD.csv


$BRK: possibly delisted; no price data found  (1d 2021-07-01 -> 2024-07-01)

1 Failed download:
['BRK']: possibly delisted; no price data found  (1d 2021-07-01 -> 2024-07-01)


[3/10] BRK: saved 753 rows via BRK-B -> BRK.csv
[4/10] DIS: saved 753 rows -> DIS.csv
[5/10] GOOG: saved 753 rows -> GOOG.csv
[6/10] MSFT: saved 753 rows -> MSFT.csv
[7/10] NVDA: saved 753 rows -> NVDA.csv
[8/10] TSLA: saved 753 rows -> TSLA.csv
[9/10] WMT: saved 753 rows -> WMT.csv
[10/10] XOM: saved 753 rows -> XOM.csv

Download complete
Successfully saved: 10/10


## 4. Cost Estimation for LLM Training on FNSPID

In [29]:
from pathlib import Path
import pandas as pd

save_dir = Path(save_dir) if 'save_dir' in globals() else Path(FNSPID_PATH)

for csv_name in ['train_df.csv', 'test_df.csv']:
    csv_path = save_dir / csv_name
    df = pd.read_csv(csv_path)

    if 'stock_symbol' in df.columns:
        df = df.rename(columns={'stock_symbol': 'label'})
    elif 'Stock_symbol' in df.columns:
        df = df.rename(columns={'Stock_symbol': 'label'})

    if 'label' not in df.columns:
        raise KeyError(f"Could not find a stock symbol column to relabel in {csv_path}")

    df.to_csv(csv_path, index=False)
    print(f"Saved relabeled dataset: {csv_path}")

Saved relabeled dataset: /home/michaelschlee/ownCloud/GIT/LabelFusion/Dataset_Descriptives/data/FNSPID/train_df.csv
Saved relabeled dataset: /home/michaelschlee/ownCloud/GIT/LabelFusion/Dataset_Descriptives/data/FNSPID/test_df.csv
